<a href="https://colab.research.google.com/github/ManuJLV/PracticasIA/blob/main/DL_text.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from keras.preprocessing import sequence
import tensorflow_datasets as ds
import matplotlib.pyplot as plt
from keras.models import Sequential
from keras.layers import Dense, Embedding
from keras.layers import LSTM
import numpy as np
import tensorflow_hub as hub


In [ ]:
batch_size = 64

In [ ]:
(train_data, test_data), info = ds.load(name="ag_news_subset", split=["train", "test"],
                                  batch_size=-1, as_supervised=True, with_info=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...:   0%|          | 0/120000 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/ag_news_subset/1.0.0.incomplete7EWOZ2/ag_news_subset-train.tfrecord*...:  …

Generating test examples...:   0%|          | 0/7600 [00:00<?, ? examples/s]

Shuffling /root/tensorflow_datasets/ag_news_subset/1.0.0.incomplete7EWOZ2/ag_news_subset-test.tfrecord*...:   …

Dataset ag_news_subset downloaded and prepared to /root/tensorflow_datasets/ag_news_subset/1.0.0. Subsequent calls will reuse this data.


In [ ]:
print(info.features["label"].names)

['World', 'Sports', 'Business', 'Sci/Tech']


In [ ]:
train_examples, train_labels = ds.as_numpy(train_data)
test_examples, test_labels = ds.as_numpy(test_data)

In [ ]:
print('x_train shape:', train_examples.shape)
print('x_test shape:', test_examples.shape)

x_train shape: (120000,)
x_test shape: (7600,)


In [ ]:
print(train_examples)

[b'AMD #39;s new dual-core Opteron chip is designed mainly for corporate computing applications, including databases, Web services, and financial transactions.'
 b'Reuters - Major League Baseball\\Monday announced a decision on the appeal filed by Chicago Cubs\\pitcher Kerry Wood regarding a suspension stemming from an\\incident earlier this season.'
 b'President Bush #39;s  quot;revenue-neutral quot; tax reform needs losers to balance its winners, and people claiming the federal deduction for state and local taxes may be in administration planners #39; sights, news reports say.'
 ...
 b'And lo, the hawk begat the dove. At least that is how one imagines Private Eye might soon be recording the remarkable events unfolding in Gaza in The Book of Sharon, its inimitable account of the travails of the modern Holy Land.'
 b'West Palm Beach, FL (Sports Network) - Tom Lehman will be named the United States Ryder Cup captain for the 2006 matches, according to The Palm Beach Post.'
 b'If the Fede

In [ ]:
print(train_labels)
train_labels = (train_labels-np.min(train_labels))/(np.max(train_labels)-np.min(train_labels))
print(train_labels)

[3 1 2 ... 0 1 2]
[1.         0.33333333 0.66666667 ... 0.         0.33333333 0.66666667]


In [ ]:
print(test_labels)
test_labels = (test_labels-np.min(test_labels))/(np.max(test_labels)-np.min(test_labels))
print(test_labels)

[1 0 3 ... 3 2 1]
[0.33333333 0.         1.         ... 1.         0.66666667 0.33333333]


In [ ]:
model = "https://tfhub.dev/google/tf2-preview/nnlm-en-dim50/1"

hub_layer = hub.KerasLayer(model, output_shape=[50], input_shape=[],
                           dtype=tf.string, trainable=True)
hub_layer(train_examples[:3])


<tf.Tensor: shape=(3, 50), dtype=float32, numpy=
array([[ 0.13279007,  0.06140124,  0.1747397 , -0.01384087, -0.00910476,
        -0.03726622,  0.07974008,  0.08505542, -0.15469442, -0.07710762,
        -0.5860853 ,  0.38640746, -0.17650622, -0.12226384,  0.22213276,
         0.37066036,  0.01225427,  0.11542831,  0.20145267,  0.16040364,
         0.03724775, -0.1422108 ,  0.04388927,  0.00514791, -0.22648726,
        -0.10230689,  0.06203717,  0.09426294,  0.04055819,  0.18911201,
         0.2816111 , -0.09024968,  0.04349989, -0.30649066,  0.20486301,
        -0.39136994,  0.25492623, -0.06430516,  0.16803294, -0.0635931 ,
         0.09554254, -0.05217019, -0.10079663,  0.259143  , -0.16179433,
        -0.18240969,  0.05787944,  0.00377896,  0.1353013 ,  0.35294548],
       [ 0.2955813 ,  0.134404  ,  0.09672645,  0.1042643 , -0.14633738,
         0.21999091, -0.2732706 ,  0.056431  ,  0.3784275 , -0.14614943,
         0.0726692 ,  0.12335153,  0.07059986, -0.2501442 ,  0.3267967 ,
 

In [ ]:
model = tf.keras.Sequential()
model.add(hub_layer)
tf.keras.layers.LSTM(16, dropout=0.2, recurrent_dropout=0.2,return_sequences=True)
model.add(tf.keras.layers.Dense(1, activation='relu'))

model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 keras_layer (KerasLayer)    (None, 50)                48190600  
                                                                 
 dense (Dense)               (None, 1)                 51        
                                                                 
Total params: 48,190,651
Trainable params: 48,190,651
Non-trainable params: 0
_________________________________________________________________


In [ ]:
model.compile(loss='binary_crossentropy',
              optimizer='adam',
              metrics=['accuracy'])

In [ ]:
train_examples, train_labels = ds.as_numpy(train_data)
test_examples, test_labels = ds.as_numpy(test_data)

print('Train...')
model.fit(train_examples, train_labels,
          batch_size=batch_size,
          epochs=10,
          validation_data=(test_examples, test_labels))
score, acc = model.evaluate(test_examples, test_labels,
                            batch_size=batch_size)
print('Test score:', score)
print('Test accuracy:', acc)

Train...
Epoch 1/10
1875/1875 [==============================] - 910s 484ms/step - loss: -9.4376 - accuracy: 0.3867 - val_loss: -10.2680 - val_accuracy: 0.4133
Epoch 2/10
1875/1875 [==============================] - 918s 490ms/step - loss: -10.4806 - accuracy: 0.4318 - val_loss: -10.3480 - val_accuracy: 0.4311
Epoch 3/10
1875/1875 [==============================] - 925s 493ms/step - loss: -10.6092 - accuracy: 0.4480 - val_loss: -10.3192 - val_accuracy: 0.4443
Epoch 4/10
1875/1875 [==============================] - 918s 489ms/step - loss: -10.7048 - accuracy: 0.4544 - val_loss: -10.4096 - val_accuracy: 0.4422
Epoch 5/10
1875/1875 [==============================] - 919s 490ms/step - loss: -10.7969 - accuracy: 0.4576 - val_loss: -10.3760 - val_accuracy: 0.4391
Epoch 6/10
1875/1875 [==============================] - 918s 490ms/step - loss: -10.8241 - accuracy: 0.4601 - val_loss: -10.3466 - val_accuracy: 0.4417
Epoch 7/10
1875/1875 [==============================] - 918s 490ms/step - loss: 